# By using lorentzian function we can get a clean spectra looking function in which we add randomness and noise in order to create entry for our dataset

In [15]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNKCIE
# ─────────────────────────────────────────────────────────────────────────────

def slugify(name: str) -> str:
    """Premení názov zlúčeniny na bezpečný názov priečinka."""
    name = name.strip().lower()
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"[^a-z0-9_\-\.]", "", name)
    return name or "unknown"


def lorentzian(x: np.ndarray, center: float, hwhm: float) -> np.ndarray:
    """
    Lorentzova funkcia — tvar jedného NMR píku.

    Parametre:
        x      : os ppm hodnôt
        center : poloha píku v ppm
        hwhm   : pološírka píku (Half Width at Half Maximum) v ppm
    """
    return (hwhm ** 2) / ((x - center) ** 2 + hwhm ** 2)


def resample_to(vector: np.ndarray, target_len: int = 1024) -> np.ndarray:
    """Resampľuje vektor na požadovanú dĺžku pomocou lineárnej interpolácie."""
    x_old = np.linspace(0, 1, len(vector))
    x_new = np.linspace(0, 1, target_len)
    return np.interp(x_new, x_old, vector).astype(np.float32)


def safe_float(value, default: float) -> float:
    """Bezpečne pretypuje hodnotu na float, inak vráti default."""
    if pd.isna(value):
        return default
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def resolve_existing_path(*candidates: str) -> str:
    """Vráti prvú existujúcu cestu zo zoznamu kandidátov."""
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(f"Nenašiel som žiadnu z ciest: {candidates}")


# ─────────────────────────────────────────────────────────────────────────────
# NASTAVENIA PRE JEDNOTLIVÉ JADRÁ
# ─────────────────────────────────────────────────────────────────────────────

NUCLEUS_SETTINGS = {
    "1H": {
        "ppm_min": -0.5,
        "ppm_max": 10.5,
        "spectrometer_mhz": 400.0,
        "global_shift": 0.03,
        "local_shift_sigma": 0.002,
        "hwhm_range": (0.0015, 0.0060),
        "intensity_scale": (0.60, 1.40),
        "noise_sigma": (0.00005, 0.00050),
        "baseline_amplitude": (0.0, 0.0030),
        "contaminants": [
            {"name": "water", "prob": 0.40, "ppm": 4.70, "ppm_sigma": 0.02, "intensity": (0.01, 0.08), "hwhm": (0.0020, 0.0200)},
            {"name": "cdcl3", "prob": 0.40, "ppm": 7.26, "ppm_sigma": 0.01, "intensity": (0.01, 0.06), "hwhm": (0.0010, 0.0100)},
        ],
    },
    "13C": {
        "ppm_min": -5.0,
        "ppm_max": 220.0,
        "spectrometer_mhz": 100.0,
        "global_shift": 0.08,
        "local_shift_sigma": 0.010,
        "hwhm_range": (0.0150, 0.0800),
        "intensity_scale": (0.85, 1.15),
        "noise_sigma": (0.00003, 0.00030),
        "baseline_amplitude": (0.0, 0.0020),
        "contaminants": [],
    },
}


def get_nucleus_settings(nucleus: str) -> dict:
    """Vráti konfiguračný slovník pre zvolené jadro."""
    nucleus = str(nucleus).strip()
    if nucleus not in NUCLEUS_SETTINGS:
        raise ValueError(f"Nepodporované jadro: {nucleus}. Podporované: {list(NUCLEUS_SETTINGS)}")
    return NUCLEUS_SETTINGS[nucleus]


# ─────────────────────────────────────────────────────────────────────────────
# J-COUPLING
# ─────────────────────────────────────────────────────────────────────────────

def get_multiplet_lines(
    center_ppm:       float,
    n_neighbors:      int,
    j_hz:             float,
    spectrometer_mhz: float = 400.0,
):
    """
    Vypočíta polohy a relatívne intenzity čiar multipletu.

    Pre 1H používa Pascalov trojuholník.
    Pre 13C v bežnom broadband decoupled režime je n_neighbors = 0,
    takže výsledok je singlet.
    """

    pascal = [
        [1],
        [1, 1],
        [1, 2, 1],
        [1, 3, 3, 1],
        [1, 4, 6, 4, 1],
        [1, 5, 10, 10, 5, 1],
    ]

    n = max(0, min(int(n_neighbors), len(pascal) - 1))
    relative_intensities = np.array(pascal[n], dtype=np.float32)
    relative_intensities /= relative_intensities.sum()

    n_lines = len(relative_intensities)
    spacing_ppm = 0.0 if j_hz <= 0 else j_hz / spectrometer_mhz

    offsets = np.linspace(
        -(n_lines - 1) / 2 * spacing_ppm,
        (n_lines - 1) / 2 * spacing_ppm,
        n_lines,
    )
    positions = center_ppm + offsets

    return positions, relative_intensities


# ─────────────────────────────────────────────────────────────────────────────
# GENERÁTOR SPEKTRA — čistá zlúčenina
# ─────────────────────────────────────────────────────────────────────────────

def generate_spectrum(
    peaks_ppm:        np.ndarray,
    peaks_int:        np.ndarray,
    peaks_neighbors:  np.ndarray,
    nucleus:          str = "1H",
    j_hz_values:      np.ndarray | None = None,
    ppm_min:          float | None = None,
    ppm_max:          float | None = None,
    n_points:         int = 8192,
    rng:              np.random.Generator | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Vygeneruje jedno syntetické NMR spektrum jednej zlúčeniny.

    Podporované jadrá:
        - 1H  : používa splitting podľa n_neighbors a typické J-couplingy
        - 13C : defaultne generuje decoupled singlets bez splittingu
    """

    rng = np.random.default_rng() if rng is None else rng
    cfg = get_nucleus_settings(nucleus)

    ppm_min = cfg["ppm_min"] if ppm_min is None else ppm_min
    ppm_max = cfg["ppm_max"] if ppm_max is None else ppm_max

    x = np.linspace(ppm_min, ppm_max, n_points).astype(np.float32)
    y = np.zeros_like(x)

    global_shift = rng.uniform(-cfg["global_shift"], cfg["global_shift"])
    scale_min, scale_max = cfg["intensity_scale"]
    global_scale = rng.uniform(scale_min, scale_max)

    if j_hz_values is None:
        j_hz_values = np.zeros(len(peaks_ppm), dtype=np.float32)

    for peak_ppm, peak_int, n_neighbors, peak_j in zip(peaks_ppm, peaks_int, peaks_neighbors, j_hz_values):
        shifted_ppm = peak_ppm + global_shift + rng.normal(0, cfg["local_shift_sigma"])
        scaled_int = peak_int * global_scale * rng.uniform(0.85, 1.15)
        hwhm = rng.uniform(*cfg["hwhm_range"])

        if nucleus == "1H":
            if int(n_neighbors) > 0:
                base_j = safe_float(peak_j, rng.uniform(3.0, 10.0))
            else:
                base_j = 0.0
            effective_neighbors = int(n_neighbors)
        else:
            base_j = 0.0
            effective_neighbors = 0

        line_positions, line_intensities = get_multiplet_lines(
            center_ppm=shifted_ppm,
            n_neighbors=effective_neighbors,
            j_hz=base_j,
            spectrometer_mhz=cfg["spectrometer_mhz"],
        )

        for line_ppm, line_rel_int in zip(line_positions, line_intensities):
            y += scaled_int * line_rel_int * lorentzian(x, line_ppm, hwhm)

    for contaminant in cfg["contaminants"]:
        if rng.random() < contaminant["prob"]:
            peak_ppm = contaminant["ppm"] + rng.normal(0, contaminant["ppm_sigma"])
            peak_int = rng.uniform(*contaminant["intensity"])
            peak_hwhm = rng.uniform(*contaminant["hwhm"])
            y += peak_int * lorentzian(x, peak_ppm, peak_hwhm)

    baseline_amplitude = rng.uniform(*cfg["baseline_amplitude"])
    raw_noise = rng.normal(0, 1, n_points)
    smoothing_kernel = np.ones(200) / 200
    smooth_baseline = np.convolve(raw_noise, smoothing_kernel, mode="same")
    y += baseline_amplitude * smooth_baseline

    noise_sigma = rng.uniform(*cfg["noise_sigma"])
    y += rng.normal(0, noise_sigma, size=n_points).astype(np.float32)

    y = y - np.median(y)
    p995 = np.percentile(np.abs(y), 99.5)
    y = np.clip(y, -p995, p995)
    y = y / (np.max(np.abs(y)) + 1e-6)

    return x, y.astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# GENERÁTOR ZMESI — viacero zlúčenín naraz (multi-label)
# ─────────────────────────────────────────────────────────────────────────────

def generate_mixture(
    all_compounds: dict,
    nucleus:       str = "1H",
    n_components:  int = 2,
    ppm_min:       float | None = None,
    ppm_max:       float | None = None,
    n_points:      int = 8192,
    rng:           np.random.Generator | None = None,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Vygeneruje spektrum zmesi viacerých zlúčenín rovnakého jadra.

    all_compounds má tvar:
        {
            compound_name: {
                "ppm": ...,
                "intensity": ...,
                "n_neighbors": ...,
                "j_hz_typical": ...,
            }
        }
    """

    rng = np.random.default_rng() if rng is None else rng
    compound_names = list(all_compounds.keys())

    if len(compound_names) < n_components:
        raise ValueError(f"Pre zmesi potrebuješ aspoň {n_components} zlúčeniny pre jadro {nucleus}.")

    chosen = rng.choice(compound_names, size=n_components, replace=False).tolist()

    cfg = get_nucleus_settings(nucleus)
    ppm_min = cfg["ppm_min"] if ppm_min is None else ppm_min
    ppm_max = cfg["ppm_max"] if ppm_max is None else ppm_max

    x = np.linspace(ppm_min, ppm_max, n_points).astype(np.float32)
    y_mixed = np.zeros(n_points, dtype=np.float32)
    ratios = rng.dirichlet(np.ones(n_components)).astype(np.float32)

    for compound_name, ratio in zip(chosen, ratios):
        compound_data = all_compounds[compound_name]
        _, y_single = generate_spectrum(
            peaks_ppm=compound_data["ppm"],
            peaks_int=compound_data["intensity"],
            peaks_neighbors=compound_data["n_neighbors"],
            nucleus=nucleus,
            j_hz_values=compound_data["j_hz_typical"],
            ppm_min=ppm_min,
            ppm_max=ppm_max,
            n_points=n_points,
            rng=rng,
        )
        y_mixed += ratio * y_single

    y_mixed = y_mixed - np.median(y_mixed)
    p995 = np.percentile(np.abs(y_mixed), 99.5)
    y_mixed = np.clip(y_mixed, -p995, p995)
    y_mixed = y_mixed / (np.max(np.abs(y_mixed)) + 1e-6)

    return x, y_mixed.astype(np.float32), chosen


# ─────────────────────────────────────────────────────────────────────────────
# ULOŽENIE PNG
# ─────────────────────────────────────────────────────────────────────────────

def save_png(
    x: np.ndarray,
    y: np.ndarray,
    path_png: str,
    dpi: int = 100,
    linewidth: float = 4.0,
):
    """
    Uloží spektrum ako PNG obrázok.

    Dôležité detaily:
        - invert_xaxis: NMR konvencia — ppm rastie sprava doľava
        - subplots_adjust: graf vyplní celú plochu bez okrajov
        - pevné y limity kvôli konzistentnému exportu obrázkov
    """
    fig, ax = plt.subplots(figsize=(10.24, 2.46))
    ax.plot(x, y, linewidth=linewidth, color="black")
    ax.invert_xaxis()
    ax.set_ylim(-0.2, 1.1)
    ax.set_xlim(x.min(), x.max())
    ax.axis("off")
    fig.patch.set_facecolor("white")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.savefig(path_png, dpi=dpi, facecolor="white")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# HLAVNÝ BUILDER
# ─────────────────────────────────────────────────────────────────────────────

def build_dataset(
    csv_path:             str,
    out_dir:              str = "dataset_out",
    nucleus:              str = "1H",
    samples_per_compound: int = 500,
    save_vectors:         bool = True,
    save_images:          bool = True,
    vector_len:           int = 1024,
    generate_mixtures:    bool = True,
    mixture_ratio:        float = 0.3,
    max_components:       int = 3,
    include_only_default: bool = True,
    seed:                 int = 0,
):
    """
    Vygeneruje celý dataset zo vstupného CSV súboru.

    Podporované vstupy:
        - starý 1H CSV formát: compound, ppm, intensity, n_neighbors
        - nový combined CSV: obsahuje aj stĺpec nucleus a ďalšie metadata

    Dôležité:
        - zachováva pôvodnú štruktúru priečinkov
        - rozlišuje výstupy podľa jadra (1H / 13C)
    """

    rng = np.random.default_rng(seed)
    df = pd.read_csv(csv_path)

    required_columns = {"compound", "ppm", "intensity", "n_neighbors"}
    if not required_columns.issubset(df.columns):
        raise ValueError(f"CSV musí obsahovať: {required_columns} (nájdené: {set(df.columns)})")

    if "nucleus" not in df.columns:
        df["nucleus"] = "1H"
    if "generator_include_default" not in df.columns:
        df["generator_include_default"] = True
    if "j_hz_typical" not in df.columns:
        df["j_hz_typical"] = np.nan

    df = df.dropna(subset=["compound", "ppm", "intensity", "n_neighbors"]).copy()
    df["compound"] = df["compound"].astype(str)
    df["nucleus"] = df["nucleus"].astype(str).str.strip()

    df = df[df["nucleus"] == nucleus].copy()
    if include_only_default:
        df = df[df["generator_include_default"] == True].copy()

    if df.empty:
        raise ValueError(
            f"Po filtrovaní nezostali žiadne riadky pre nucleus={nucleus!r}. "
            f"Skontroluj CSV alebo include_only_default=False."
        )

    os.makedirs(out_dir, exist_ok=True)

    compounds = sorted(df["compound"].unique().tolist())
    label_map = {compound: idx for idx, compound in enumerate(compounds)}

    with open(os.path.join(out_dir, "label_map.json"), "w", encoding="utf-8") as f:
        json.dump(label_map, f, ensure_ascii=False, indent=2)

    all_compounds = {}
    for compound, group in df.groupby("compound"):
        all_compounds[compound] = {
            "ppm": group["ppm"].to_numpy(np.float32),
            "intensity": group["intensity"].to_numpy(np.float32),
            "n_neighbors": group["n_neighbors"].fillna(0).to_numpy(int),
            "j_hz_typical": group["j_hz_typical"].to_numpy(np.float32),
        }

    if save_vectors:
        os.makedirs(os.path.join(out_dir, "vectors"), exist_ok=True)
        if generate_mixtures:
            os.makedirs(os.path.join(out_dir, "vectors", "_mixtures"), exist_ok=True)
    if save_images:
        os.makedirs(os.path.join(out_dir, "images"), exist_ok=True)
        if generate_mixtures:
            os.makedirs(os.path.join(out_dir, "images", "_mixtures"), exist_ok=True)

    rows = []
    cfg = get_nucleus_settings(nucleus)

    for compound, group in df.groupby("compound", sort=False):
        peaks_ppm = group["ppm"].to_numpy(np.float32)
        peaks_int = group["intensity"].to_numpy(np.float32)
        peaks_neighbors = group["n_neighbors"].fillna(0).to_numpy(int)
        j_hz_values = group["j_hz_typical"].to_numpy(np.float32)
        folder_name = slugify(compound)

        vec_dir = os.path.join(out_dir, "vectors", folder_name)
        img_dir = os.path.join(out_dir, "images", folder_name)

        if save_vectors:
            os.makedirs(vec_dir, exist_ok=True)
        if save_images:
            os.makedirs(img_dir, exist_ok=True)

        n_pure = int(samples_per_compound * (1 - mixture_ratio)) if generate_mixtures else samples_per_compound

        for i in range(n_pure):
            x, y = generate_spectrum(
                peaks_ppm=peaks_ppm,
                peaks_int=peaks_int,
                peaks_neighbors=peaks_neighbors,
                nucleus=nucleus,
                j_hz_values=j_hz_values,
                ppm_min=cfg["ppm_min"],
                ppm_max=cfg["ppm_max"],
                rng=rng,
            )

            y_final = resample_to(y, target_len=vector_len)
            x_final = np.linspace(x.min(), x.max(), vector_len)

            sample_id = f"{folder_name}_{i:05d}"
            png_path = ""
            npy_path = ""

            if save_images:
                png_path = os.path.join(img_dir, f"{sample_id}.png")
                save_png(x_final, y_final, png_path)

            if save_vectors:
                npy_path = os.path.join(vec_dir, f"{sample_id}.npy")
                np.save(npy_path, y_final)

            rows.append({
                "sample_id": sample_id,
                "compound": compound,
                "label": label_map[compound],
                "nucleus": nucleus,
                "is_mixture": False,
                "components": compound,
                "png_path": png_path,
                "npy_path": npy_path,
            })

        print(f"{nucleus} | čisté: {compound} → {n_pure} vzoriek")

    if generate_mixtures and len(compounds) >= 2:
        n_mixtures = int(samples_per_compound * mixture_ratio * len(compounds))
        mix_vec_dir = os.path.join(out_dir, "vectors", "_mixtures")
        mix_img_dir = os.path.join(out_dir, "images", "_mixtures")

        for i in range(n_mixtures):
            n_components = int(rng.integers(2, min(max_components, len(compounds)) + 1))
            x, y_mix, chosen = generate_mixture(
                all_compounds=all_compounds,
                nucleus=nucleus,
                n_components=n_components,
                ppm_min=cfg["ppm_min"],
                ppm_max=cfg["ppm_max"],
                rng=rng,
            )

            y_final = resample_to(y_mix, target_len=vector_len)
            x_final = np.linspace(x.min(), x.max(), vector_len)
            sample_id = f"mixture_{i:05d}"
            png_path = ""
            npy_path = ""

            if save_images:
                png_path = os.path.join(mix_img_dir, f"{sample_id}.png")
                save_png(x_final, y_final, png_path)

            if save_vectors:
                npy_path = os.path.join(mix_vec_dir, f"{sample_id}.npy")
                np.save(npy_path, y_final)

            component_labels = [label_map[c] for c in chosen]

            rows.append({
                "sample_id": sample_id,
                "compound": "+".join(chosen),
                "label": str(component_labels),
                "nucleus": nucleus,
                "is_mixture": True,
                "components": "+".join(chosen),
                "png_path": png_path,
                "npy_path": npy_path,
            })

        print(f"\n{nucleus} | zmesi → {n_mixtures} vzoriek")

    index_df = pd.DataFrame(rows)
    index_df.to_csv(os.path.join(out_dir, "index.csv"), index=False)

    pure_count = len([r for r in rows if not r["is_mixture"]])
    mixture_count = len([r for r in rows if r["is_mixture"]])

    print(f"\nDataset uložený do: {out_dir}")
    print(f"Jadro:          {nucleus}")
    print(f"Čisté vzorky:   {pure_count}")
    print(f"Zmesi:          {mixture_count}")
    print(f"Celkovo:        {len(rows)}")
    print(f"Zlúčeniny ({len(compounds)}): {compounds}")


# ─────────────────────────────────────────────────────────────────────────────
# SPUSTENIE
# ─────────────────────────────────────────────────────────────────────────────

CSV_PATH = "../spectra_dataset.csv"

# Koreňový priečinok pre výstupy. Preferuje aktuálny priečinok projektu.
OUT_ROOT = resolve_existing_path(
    "..",
    ".",
    "/mnt/data",
)

# Zvoľ jadro:
#   "1H"  → protónové spektrá
#   "13C" → uhlíkové spektrá
NUCLEUS = "1H"

# Výstup sa uloží do samostatného priečinka podľa jadra.
OUT_DIR = os.path.join(OUT_ROOT, "nmr_dataset", NUCLEUS)

build_dataset(
    csv_path=CSV_PATH,
    out_dir=OUT_DIR,
    nucleus=NUCLEUS,
    samples_per_compound=5,
    save_vectors=True,
    save_images=True,
    vector_len=1024,
    generate_mixtures=True,
    mixture_ratio=0.3,
    max_components=3,
    include_only_default=False,
    # IB031 Klasika|
    seed=42,
)


1H | čisté: acetaldehyde → 3 vzoriek
1H | čisté: acetic_acid → 3 vzoriek
1H | čisté: acetone → 3 vzoriek
1H | čisté: acetonitrile → 3 vzoriek
1H | čisté: acetophenone → 3 vzoriek
1H | čisté: aniline → 3 vzoriek
1H | čisté: anisole → 3 vzoriek
1H | čisté: benzaldehyde → 3 vzoriek
1H | čisté: benzene → 3 vzoriek
1H | čisté: benzonitrile → 3 vzoriek
1H | čisté: bromobenzene → 3 vzoriek
1H | čisté: chlorobenzene → 3 vzoriek
1H | čisté: chloroform → 3 vzoriek
1H | čisté: cumene → 3 vzoriek
1H | čisté: cyclohexane → 3 vzoriek
1H | čisté: dichloromethane → 3 vzoriek
1H | čisté: diethyl_ether → 3 vzoriek
1H | čisté: diethylamine → 3 vzoriek
1H | čisté: dimethyl_ether → 3 vzoriek
1H | čisté: dimethyl_sulfoxide → 3 vzoriek
1H | čisté: dimethylformamide → 3 vzoriek
1H | čisté: dioxane → 3 vzoriek
1H | čisté: ethanol → 3 vzoriek
1H | čisté: ethyl_acetate → 3 vzoriek
1H | čisté: ethylbenzene → 3 vzoriek
1H | čisté: formic_acid → 3 vzoriek
1H | čisté: furan → 3 vzoriek
1H | čisté: hexane → 3 vzoriek